# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through exploring the FAIR^2 dataset using the `mlcroissant` library. You'll load metadata, examine available record sets and fields using their `@id`s, extract data, perform exploratory analysis, and visualize key aspects of the dataset.

### Dataset Source
The dataset is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Display all available record set @ids in the dataset
print("Available record sets (by @id):")
record_sets = []
for record_set in dataset.record_sets:
    print(f"  - {record_set['@id']}  (name: {record_set.get('name')})")
    record_sets.append(record_set['@id'])

# For each record set, print its fields by @id
for record_set in dataset.record_sets:
    print(f"\nFields in record set @id='{record_set['@id']}':")
    if 'field' in record_set:
        fields = record_set['field']
        # Some croissant schemas have one or many fields
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"  - {field['@id']}  (name: {field.get('name')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into DataFrames, indexed by @id
dfs = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dfs[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# If there are multiple record sets, pick the main one for EDA (choose the largest by number of columns)
main_record_set_id = None
max_cols = 0
for k, v in dfs.items():
    if v.shape[1] > max_cols:
        main_record_set_id = k
        max_cols = v.shape[1]
if main_record_set_id is None and len(dfs):
    main_record_set_id = list(dfs.keys())[0]  # fallback

print(f"\nUsing record set @id='{main_record_set_id}' for further analysis.")
print("Columns in selected DataFrame:")
print(dfs[main_record_set_id].columns.tolist())

# Show a preview of the data
dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. We will dynamically detect a numeric field and a candidate grouping field using available field `@id`s from the main record set.

In [ ]:
# Try to select a numeric field (float/integer) and a group field from the DataFrame
df = dfs[main_record_set_id]

# Try to select an integer/float field (by checking types of first row)
numeric_field = None
group_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
# Select another non-numeric field as group by if available
for col in df.columns:
    if col != numeric_field and df[col].dtype == object:
        group_field = col
        break

print(f"Selected numeric field: {numeric_field}")
print(f"Selected group field: {group_field}")

# EDA: filter by a threshold of numeric_field, normalize, group (if possible)
if numeric_field is not None:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution and grouped means (if group_field is present)
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

if group_field and group_field in df.columns and numeric_field:
    plt.figure(figsize=(10, 5))
    group_means = df.groupby(group_field)[numeric_field].mean().reset_index()
    sns.barplot(data=group_means, x=group_field, y=numeric_field)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean of '{numeric_field}' by '{group_field}'")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 tabular dataset using `mlcroissant`, starting from metadata inspection, reviewing the record set structure (via `@id`), extracting data, dynamically processing numeric and categorical fields, and visualizing key distributions. For domain-specific insights, consult the dataset documentation or data dictionary and consider further clinical/statistical analysis based on your research questions.